# SC-04 — Fonctions et état Solidity avec C#

[Version Python](SC-04-Functions-State-Python.ipynb) | [Solidity Basics](SC-03-Solidity-Basics-CSharp.ipynb) | [Héritage](SC-05-Inheritance-Python.ipynb)

**Objectifs** : distinguer storage, memory et calldata, les visibilités, les fonctions view/pure/payable et les modifiers ; mesurer les appels et manipuler des contrats financés. **Prérequis** : SC03, .NET Interactive, Nethereum 7.1.0, solc 0.8.28 et Anvil local. **Durée** : environ 2 h.

Lancer `anvil --host 127.0.0.1 --chain-id 31337` ; les transactions utilisent les comptes déverrouillés de ce bac local et attendent leur minage.

In [1]:
#r "nuget:Nethereum.Web3, 7.1.0"
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;
using System.Numerics;
using System.Text.Json;
using Nethereum.Web3;
using Nethereum.Hex.HexTypes;
using Nethereum.Contracts;

// Vérifier le compilateur demandé avant de compiler toute source.
var optionsVersion = new ProcessStartInfo("solc", "--version") { RedirectStandardOutput = true };
var version = Process.Start(optionsVersion) ?? throw new InvalidOperationException("Impossible de démarrer solc.");
string versionTexte = await version.StandardOutput.ReadToEndAsync();
await version.WaitForExitAsync();
if (version.ExitCode != 0 || !versionTexte.Contains("0.8.28+commit.7893614a"))
    throw new InvalidOperationException("solc 0.8.28 requis.");
Console.WriteLine(versionTexte.Trim());
version.Dispose();

var url = "http://127.0.0.1:8545";
var web3 = new Web3(url);
var chain = await web3.Eth.ChainId.SendRequestAsync();
if (chain.Value != 31337)
    throw new InvalidOperationException("Anvil local attendu (chaîne 31337).");

// Les comptes déverrouillés proviennent exclusivement du bac Anvil local.
var accounts = await web3.Eth.Accounts.SendRequestAsync();
var deployer = accounts[0];
Console.WriteLine($"Chaîne {chain.Value}, compte {deployer}.");

async Task<Contract> Deploy(string source, string name, params object[] args)
{
    // Standard JSON transmet le texte Solidity complet et indique les deux artefacts nécessaires.
    var artefacts = new[] { "abi", "evm.bytecode.object" };
    var fichier = new Dictionary<string, object> { ["Example.sol"] = new { content = source } };
    var selection = new Dictionary<string, object>
    {
        ["*"] = new Dictionary<string, string[]> { ["*"] = artefacts }
    };
    string input = JsonSerializer.Serialize(new
    {
        language = "Solidity",
        sources = fichier,
        settings = new { outputSelection = selection }
    });
    var options = new ProcessStartInfo("solc", "--standard-json")
    {
        RedirectStandardInput = true,
        RedirectStandardOutput = true,
        RedirectStandardError = true
    };
    using var p = Process.Start(options) ?? throw new InvalidOperationException("Impossible de compiler le contrat.");
    var stdout = p.StandardOutput.ReadToEndAsync();
    var stderr = p.StandardError.ReadToEndAsync();
    await p.StandardInput.WriteAsync(input);
    p.StandardInput.Close();
    await p.WaitForExitAsync();
    if (p.ExitCode != 0)
        throw new InvalidOperationException(await stderr);

    using var doc = JsonDocument.Parse(await stdout);
    if (doc.RootElement.TryGetProperty("errors", out var errors) &&
        errors.EnumerateArray().Any(e => e.GetProperty("severity").GetString() == "error"))
        throw new InvalidOperationException(errors.ToString());

    // Choisir le contrat nommé : le bytecode s’exécute dans l’EVM et l’ABI encode les appels du client.
    var contrats = doc.RootElement.GetProperty("contracts").GetProperty("Example.sol");
    var contrat = contrats.GetProperty(name);
    var abi = contrat.GetProperty("abi").GetRawText();
    var bin = contrat.GetProperty("evm").GetProperty("bytecode").GetProperty("object").GetString();
    if (string.IsNullOrEmpty(bin) || abi == "[]")
        throw new InvalidOperationException("ABI ou bytecode absent.");

    // Attendre le receipt de déploiement avant de construire un objet d'accès au contrat.
    var r = await web3.Eth.DeployContract.SendRequestAndWaitForReceiptAsync(
        abi, "0x" + bin, deployer, new HexBigInteger(5_000_000), null, args);
    if (r.Status?.Value != BigInteger.One || r.BlockNumber == null ||
        string.IsNullOrEmpty(r.ContractAddress) || string.IsNullOrEmpty(r.TransactionHash))
        throw new InvalidOperationException("Déploiement non miné ou sans succès.");
    Console.WriteLine($"{name} déployé : {r.ContractAddress}, tx {r.TransactionHash}, gas {r.GasUsed.Value}");
    return web3.Eth.GetContract(abi, r.ContractAddress);
}

async Task Write(Contract c, string fn, params object[] args)
{
    // Le déployeur signe dans le bac local. Le statut distingue un revert d'une écriture réussie.
    var r = await c.GetFunction(fn).SendTransactionAndWaitForReceiptAsync(
        deployer, new HexBigInteger(1_000_000), null, null, args);
    if (r.BlockNumber == null || string.IsNullOrEmpty(r.TransactionHash))
        throw new InvalidOperationException("Transaction non minée.");
    if (r.Status?.Value != BigInteger.One)
        throw new InvalidOperationException($"Receipt status 0 : {r.TransactionHash}, bloc {r.BlockNumber.Value}, gas {r.GasUsed.Value}");
    Console.WriteLine($"{fn} : tx {r.TransactionHash}, bloc {r.BlockNumber.Value}, gas {r.GasUsed.Value}");
}
Console.WriteLine("Compilation standard-json et transactions configurées.");

async Task WriteFrom(string from, Contract c, string fn, BigInteger wei, params object[] args)
{
    // value transporte une quantité de Wei entière. from choisit l'un des comptes locaux disponibles.
    var receipt = await c.GetFunction(fn).SendTransactionAndWaitForReceiptAsync(
        from, new HexBigInteger(1_000_000), new HexBigInteger(wei), null, args);
    if (receipt.Status?.Value != BigInteger.One || receipt.BlockNumber == null ||
        string.IsNullOrEmpty(receipt.TransactionHash))
        throw new InvalidOperationException("Transaction non minée ou sans succès.");
    Console.WriteLine($"{fn} : tx {receipt.TransactionHash}, bloc {receipt.BlockNumber.Value}, gas {receipt.GasUsed.Value}");
}
var eth = BigInteger.Pow(10, 18);
Console.WriteLine("Montants en Wei : 1 ETH = 10^18 Wei.");

Installed Packages Nethereum.Web3, 7.1.0

solc, the solidity compiler commandline interface
Version: 0.8.28+commit.7893614a.Linux.g++


Chaîne 31337, compte 0xf39fd6e51aad88f6f4ce6ab8827279cfffb92266.


Compilation standard-json et transactions configurées.


Montants en Wei : 1 ETH = 10^18 Wei.


***

## 1. Data Locations

Solidite définit trois emplacements de données :

| Location | Description | Coût gas |
|----------|-------------|----------|
| `storage` | Blockchain permanente | Élevé |
| `memory` | Temporaire (RAM) | Faible |
| `calldata` | Données d'entrée (read-only) | Minimal |

Le choix de la data location change le coût et la durée de vie de la donnée : `storage` persiste entre les appels (écriture coûteuse en gas), `memory` n'existe que le temps de l'appel, et `calldata` — lecture seule, non modifiable — est la source la moins chère pour les arguments externes. Règle mnémotechnique : variables d'état en `storage`, arguments de fonctions externes en `calldata`, tout le reste en `memory`.


In [2]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract DataLocations {
    // STORAGE : stocke sur la blockchain (permanent)
    uint256[] public numbers;  // storage par defaut pour variables d'etat
    mapping(address => uint256) public balances;

    // MEMORY : temporaire, efface apres l'appel
    function processArray(uint256[] memory arr) public pure returns (uint256) {
        uint256 sum = 0;
        for (uint i = 0; i < arr.length; i++) {
            sum += arr[i];
        }
        return sum;
    }

    // CALLDATA : read-only, le moins couteux
    function getElement(uint256[] calldata arr, uint256 index) 
        external pure returns (uint256) 
    {
        return arr[index];
    }

    // Storage reference pour modification
    function updateNumber(uint256 index, uint256 value) public {
        numbers[index] = value;  // Modification directe du storage
    }
}
";

 // Déployer la structure qui expose les paramètres des trois emplacements de données.
var c = await Deploy(source,"DataLocations");


}

DataLocations déployé : 0x99dbe4aea58e518c50a1c04ae9b48c9f6354612f, tx 0x11a2ebd9294ab48a2c859a0a3a28900d59eecacd513b10f6cdacf4bbe260aa3c, gas 407980


Les valeurs et les frais affichés ci-dessus proviennent des appels et receipts de cette exécution locale.

### 1.1 Règles de data location

- **Variables d'état** : toujours en `storage`
- **Arguments de fonction** : `memory` ou `calldata` pour les types complexes
- **Variables locales** : `storage` (reference) ou `memory` (valeur copiee)

Le piège classique est la copie implicite : `User memory u = users[i]` copie la struct en mémoire, et modifier `u` ne touche pas le storage — la cellule suivante le démontre concrètement.


In [3]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract StorageVsMemory {
    struct User {
        string name;
        uint256 balance;
    }

    User[] public users;

    function addUser(string memory name, uint256 balance) public {
        users.push(User(name, balance));
    }

    // STORAGE reference : modifie directement le tableau
    function updateBalanceGood(uint256 index, uint256 amount) public {
        User storage user = users[index];
        user.balance = amount;  // Modification permanente
    }

    // MEMORY copy : ne modifie PAS le storage (bug courant!)
    function updateBalanceBad(uint256 index, uint256 amount) public {
        User memory user = users[index];
        user.balance = amount;  // Modification locale uniquement!
    }

    function getBalance(uint256 index) public view returns (uint256) {
        return users[index].balance;
    }
}
";

 // Les deux mises à jour sont minées, puis le getter distingue la référence storage de la copie memory.
var c = await Deploy(source,"StorageVsMemory");

await Write(c,"addUser","Alice",new BigInteger(1000));
 Console.WriteLine($"Alice initiale = {await c.GetFunction("getBalance").CallAsync<BigInteger>(new BigInteger(0))}");
 await Write(c,"updateBalanceGood",new BigInteger(0),new BigInteger(2000));
 Console.WriteLine($"storage = {await c.GetFunction("getBalance").CallAsync<BigInteger>(new BigInteger(0))}");
 await Write(c,"updateBalanceBad",new BigInteger(0),new BigInteger(9999));
 Console.WriteLine($"après copie memory = {await c.GetFunction("getBalance").CallAsync<BigInteger>(new BigInteger(0))}");

}

StorageVsMemory déployé : 0x6f6f570f45833e249e27022648a26f4076f48f78, tx 0xf874e3bfc4e567f09311974e1aa2e7196a19e23a2e415b2c07df3f8f35411119, gas 576598


addUser : tx 0x0b2249c095f62feec7991a4419b45b7141c135e37dd5b88efd138100cf4624f6, bloc 144, gas 89939


Alice initiale = 1000


updateBalanceGood : tx 0x67c7059d52c2cbb8d2b34d92cc7446532a855e714d03522a7c0ab2bf75cb4490, bloc 145, gas 29101


storage = 2000


updateBalanceBad : tx 0xc9d23c13702c4f5013fa058288cfb7581e24acd316d14993e4de612b15b94119, bloc 146, gas 29060


après copie memory = 2000


### Interpretation : Storage vs Memory - Différence critique

**Résultat obtenu** : Une reference `storage` modifie l'état, une copie `memory` ne le fait pas.

| Fonction | Type de variable | Comportement | Résultat |
|----------|------------------|--------------|----------|
| `updateBalanceGood()` | `User storage` | Reference directe | Modifie le storage (2000) |
| `updateBalanceBad()` | `User memory` | Copie locale | Ne modifie PAS le storage (reste 2000) |

**Points clés** :
- `User storage user = users[index]` créé une REFERENCE vers le storage : les modifications sont permanentes
- `User memory user = users[index]` créé une COPIE locale : les modifications sont perdues a la fin de la fonction
- C'est un bug classique en Solidity : penser modifier une variable alors qu'on ne modifie qu'une copie
- Règle pratique : toujours utiliser `storage` pour modifier des structures complexes, `memory` pour la lecture seule
- Le coût gas est plus élevé pour `storage` mais nécessaire pour les modifications

***

## 2. Visibilite des fonctions

| Visibilite | Exterieur | Contrat enfant | Même contrat |
|------------|-----------|----------------|--------------|
| `public` | Oui | Oui | Oui |
| `external` | Oui | Non | Non |
| `internal` | Non | Oui | Oui |
| `private` | Non | Non | Oui |

La visibilité n'est pas un secret : tout le storage d'un contrat est lisible publiquement sur la blockchain, quoi que l'on déclare. Elle organise seulement qui peut appeler quoi — et le compilateur génère automatiquement un getter pour chaque variable d'état `public`.


In [4]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract VisibilityExample {
    uint256 private privateVar = 1;
    uint256 internal internalVar = 2;
    uint256 public publicVar = 3;

    function publicFunction() public pure returns (string memory) {
        return ""Je suis publique!"";
    }

    function externalFunction() external pure returns (string memory) {
        return ""Je suis externe!"";
    }

    function internalFunction() internal pure returns (string memory) {
        return ""Je suis interne!"";
    }

    function privateFunction() private pure returns (string memory) {
        return ""Je suis prive!"";
    }

    function callInternal() public pure returns (string memory) {
        return internalFunction();
    }

    function getPrivateVar() public view returns (uint256) {
        return privateVar;
    }

    function getAllVars() public view returns (uint256, uint256, uint256) {
        return (privateVar, internalVar, publicVar);
    }
}
";

 // L’ABI expose les getters publics et la méthode qui renvoie les valeurs privées ou internes.
var c = await Deploy(source,"VisibilityExample");

Console.WriteLine($"publicFunction = {await c.GetFunction("publicFunction").CallAsync<string>()}; callInternal = {await c.GetFunction("callInternal").CallAsync<string>()}; publicVar = {await c.GetFunction("publicVar").CallAsync<BigInteger>()}");
 var v=await c.GetFunction("getAllVars").CallDecodingToDefaultAsync();
 Console.WriteLine(string.Join(", ",v.Select(x=>x.Result)));

}

VisibilityExample déployé : 0x976fcd02f7c4773dd89c309fbf55d5923b4c98a1, tx 0x566d172654019c0152ae16e9102b8489fc19f4aa881de314b8284b05d66dbf61, gas 295689


publicFunction = Je suis publique!; callInternal = Je suis interne!; publicVar = 3


1, 2, 3


### Interpretation : Visibilite des fonctions

**Résultat obtenu** : Les quatre niveaux de visibilité controlent l'accès aux fonctions.

| Visibilite | Externe | Enfants | Interne | Usage |
|------------|---------|---------|---------|------|
| `public` | Oui | Oui | Oui | Fonctions générales |
| `external` | Oui | Non | Non | Optimisation gas |
| `internal` | Non | Oui | Oui | Helpers internes |
| `private` | Non | Non | Oui | Implementation privee |

**Points clés** :
- `publicFunction()` et `publicVar` sont accessibles de partout
- `externalFunction()` est optimise pour les appels externes (pas d'appel interne possible)
- `callInternal()` appelle `internalFunction()` (autorise car même contrat)
- `privateVar` est accessible via un getter public `getPrivateVar()` mais pas directement
- La visibilité limite l'accessite mais pas la sécurité (tout est visible sur la blockchain)

La copie `calldata -> memory` a un coût : pour un grand tableau passé à une fonction `public`, chaque appel recopie toutes les données en mémoire. `external` évite cette copie en gardant les arguments en `calldata` — la cellule suivante illustre le mécanisme sur `[1..5]` : les deux fonctions rendent `15`, mais seule `sumPublic` paie la copie mémoire.


In [5]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract GasOptimization {
    // EXTERNAL est moins couteux pour les tableaux
    // car les donnees restent en calldata (pas de copie)

    // Moins efficace (copie en memory)
    function sumPublic(uint256[] memory arr) public pure returns (uint256) {
        uint256 total = 0;
        for (uint i = 0; i < arr.length; i++) {
            total += arr[i];
        }
        return total;
    }

    // Plus efficace (reste en calldata)
    function sumExternal(uint256[] calldata arr) external pure returns (uint256) {
        uint256 total = 0;
        for (uint i = 0; i < arr.length; i++) {
            total += arr[i];
        }
        return total;
    }
}
";

 // Mesurer les deux fonctions sur le même tableau ABI pour isoler le trajet des données.
var c = await Deploy(source,"GasOptimization");

var a=new BigInteger[]{1,2,3,4,5};
 var pub=c.GetFunction("sumPublic");
 var ext=c.GetFunction("sumExternal");
 Console.WriteLine($"public = {await pub.CallAsync<BigInteger>(new object[]{a})}; external = {await ext.CallAsync<BigInteger>(new object[]{a})}");
 Console.WriteLine($"Estimation gas, mêmes entrées : public = {(await pub.EstimateGasAsync(new object[]{a})).Value}; external = {(await ext.EstimateGasAsync(new object[]{a})).Value}");

}

GasOptimization déployé : 0x19ceccd6942ad38562ee10bafd44776ceb67e923, tx 0xcf7739c1c44d25ace65551f1f1ae7e4f38e9646ba735aee6e37de11375269486, gas 310017


public = 15; external = 15


Estimation gas, mêmes entrées : public = 25761; external = 24286


### Interpretation : External vs Public - Optimisation gas

**Résultat obtenu** : Sur ce contrat et ces entrées, comparez les estimations affichées pour les deux fonctions.

| Fonction | Type de paramètre | Copie mémoire ? | Coût gas |
|----------|------------------|-----------------|----------|
| `sumPublic()` | `memory` | Oui | Plus élevé |
| `sumExternal()` | `calldata` | Non | Moins élevé |

**Points clés** :
- `public` copie les tableaux de `calldata` vers `memory` (allocation et copie couteuse)
- `external` garde les données en `calldata` (read-only, pas de copie)
- La différence de gas devient significative pour les grands tableaux
- `external` ne peut pas être appele depuis l'interieur du contrat (pas d'appel interne)
- Règle pratique : utiliser `external` pour les fonctions appeles de l'exterieur avec des paramètres complexes (tableaux, strings)

***

## 3. Modificateurs de fonction

### 3.1 view et pure

Deux familles de modificateurs : ceux qui bornent l'accès au storage (`view` lit sans écrire, `pure` ne touche pas au storage) et ceux qui gèrent les flux d'ETH (`payable`). Conséquence pratique : un appel `view`/`pure` se fait en lecture locale, gratuitement, via `.call()`, tandis que toute écriture reste une transaction qui consomme du gas.


In [6]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract ViewPureExample {
    uint256 public value = 42;

    // VIEW : lit le storage mais ne modifie pas
    function getValue() public view returns (uint256) {
        return value;
    }

    // PURE : ni lecture ni ecriture du storage
    function add(uint256 a, uint256 b) public pure returns (uint256) {
        return a + b;
    }

    // Fonction normale : peut lire et ecrire
    function increment() public returns (uint256) {
        value += 1;
        return value;
    }
}
";

 // Les appels view et pure se lisent par eth_call. Increment passe par une écriture minée.
var c = await Deploy(source,"ViewPureExample");

Console.WriteLine($"getValue = {await c.GetFunction("getValue").CallAsync<BigInteger>()}; add = {await c.GetFunction("add").CallAsync<BigInteger>(new BigInteger(10),new BigInteger(32))}");
 await Write(c,"increment");
 Console.WriteLine($"après increment = {await c.GetFunction("getValue").CallAsync<BigInteger>()}");

}

ViewPureExample déployé : 0xd42912755319665397ff090fbb63b1a31ae87cee, tx 0x39f1f1ca403d01e7badee03b141800e99748607ff396d61b59891e8360fa41ae, gas 202432


getValue = 42; add = 42


increment : tx 0x7310bdcf2b9ce6a313326bef986e79c72a7a54313ed65315e52dde2e841ae1e8, bloc 150, gas 26749


après increment = 43


### Interpretation : View et Pure - Optimisation gas

**Résultat obtenu** : Les modificateurs `view` et `pure` indiquent si une fonction accede au storage.

| Modificateur | Lit le storage ? | Modifie le storage ? | Coût gas |
|--------------|------------------|---------------------|----------|
| (aucun) | Possible | Possible | Élevé |
| `view` | Oui | Non | Faible |
| `pure` | Non | Non | Minimal |

**Points clés** :
- `getValue()` est `view` : lit `value` mais ne la modifie pas
- `add(10, 32)` est `pure` : ni lecture ni ecriture du storage (calcul local)
- `increment()` est normale : modifie `value` (ecriture dans le storage)
- Les fonctions `view` et `pure` peuvent être executees hors-chaîne (sans gas) par un client ETH
- Le compilateur vérifie que les modificateurs sont corrects (erreur si une fonction `pure` lit une variable d'état)

### Exercice 1 : Contrat Calculator (view vs pure)

Distinguez les fonctions **view** et **pure** (section 3.1). Une calculatrice
qui montre la différence entre lecture seule et aucun accès storage.

**Indice** : `pure` = ni lecture ni ecriture storage. `view` = lecture seule.

Compétence visée : distinguer `pure` (calcul local, sans storage — cf. `add` de la section 3.1), `view` (lecture seule du storage — cf. `getValue`) et une fonction qui écrit réellement dans le storage.


In [7]:
{
// Exercice 1 : Contrat Calculator (view vs pure)
// TODO etudiant : implementer une calculatrice distinguant pure / view / ecriture
// Etape 1 : Definir uint256 public lastResult
// Etape 2 : Implementer add(uint256 a, uint256 b) public pure returns (uint256)
// Etape 3 : Implementer computeAndStore(uint256 a, uint256 b) public returns (uint256)
// Etape 4 : Implementer getLast() public view returns (uint256)
var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Calculator {
    // TODO etudiant : lastResult, add (pure), computeAndStore, getLast (view)
}
";

// var c = await Deploy(source, "Calculator");
// Console.WriteLine(await c.GetFunction("add").CallAsync<BigInteger>(new BigInteger(2), new BigInteger(3)));
// await Write(c, "computeAndStore", new BigInteger(10), new BigInteger(5));
// Console.WriteLine(await c.GetFunction("getLast").CallAsync<BigInteger>());
Console.WriteLine("Exercice à compléter : Calculator");

}

Exercice à compléter : Calculator



(8,5): warning CS0219: The variable 'source' is assigned but its value is never used



### 3.2 payable

Seule une fonction déclarée `payable` — ou `receive()` / `fallback()` — peut recevoir des ETH : `msg.value` porte alors le montant envoyé avec l'appel. Sans `payable`, l'EVM rejette purement et simplement le transfert.


In [8]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract PayableExample {
    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        owner = msg.sender;
    }

    // Fonction payable : peut recevoir des ETH
    function deposit() public payable {
        balances[msg.sender] += msg.value;
    }

    // Retirer des ETH
    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, ""Fonds insuffisants"");
        balances[msg.sender] -= amount;
        (bool success, ) = payable(msg.sender).call{value: amount}("""");
        require(success, ""Echec du transfert"");
    }

    // Consulter le solde du contrat
    function getContractBalance() public view returns (uint256) {
        return address(this).balance;
    }

    // Receive : appele lors d'un transfert ETH simple
    receive() external payable {
        balances[msg.sender] += msg.value;
    }
}
";

 // Envoyer la valeur payable en Wei entiers, puis vérifier séparément le solde du contrat et le crédit comptable.
var c = await Deploy(source,"PayableExample");

Console.WriteLine($"solde initial = {await c.GetFunction("getContractBalance").CallAsync<BigInteger>()} Wei");
 await WriteFrom(deployer,c,"deposit",eth);
 Console.WriteLine($"contrat = {await c.GetFunction("getContractBalance").CallAsync<BigInteger>()}; déposant = {await c.GetFunction("balances").CallAsync<BigInteger>(deployer)} Wei");
 await Write(c,"withdraw",eth/2);
 Console.WriteLine($"après retrait = {await c.GetFunction("getContractBalance").CallAsync<BigInteger>()} Wei");

}

PayableExample déployé : 0x927b167526babb9be047421db732c663a0b77b11, tx 0xac8eba7cfc622e10691ecb853d4c9bbb02a882aa335b9f5d6bf18acf018b41ec, gas 442078


solde initial = 0 Wei


deposit : tx 0x94f66b3f7540bb508576f04a6306a96b628535ad434878b0b1068c9eee2702e2, bloc 152, gas 43646


contrat = 1000000000000000000; déposant = 1000000000000000000 Wei


withdraw : tx 0xb4f2a71d6a65f1ee7d5e45b4af0b95483b5f770eedca40a13a98fd5264e2dafd, bloc 153, gas 34279


après retrait = 500000000000000000 Wei


### Interpretation : Payable - Gestion des ETH

**Résultat obtenu** : Le mot-clé `payable` permet aux fonctions de recevoir des ETH.

| Élément | Valeur | Signification |
|---------|--------|---------------|
| Balance initiale | 0 ETH | Contrat vide au déploiement |
| Après deposit(1 ETH) | 1 ETH | Le contrat a recu 1 ETH |
| Après withdraw(0.5 ETH) | 0.5 ETH | Retrait de 0.5 ETH vers le deployer |

**Points clés** :
- `deposit()` est `payable` : peut recevoir de l'ETH via `msg.value`
- `balances[msg.sender]` suit les depots de chaque utilisateur
- `withdraw()` vérifie les fonds (`require`), debite le solde, et transfere l'ETH
- Le pattern `payable(addr).call{value: amount}("")` est la méthode moderne de transfert (preferable a `transfer()` ou `send()`)
- `receive()` est une fonction speciale appelee automatiquement lors d'un transfert ETH simple (sans data)

***

## 4. Custom Modifiers

Les modifiers permettent de reutiliser des conditions prealables.

Un modifier s'écrit comme une fonction dont le corps est remplacé par `_;` : le code placé avant le `_;` s'exécute avant la fonction, celui placé après s'exécute après. C'est le mécanisme standard pour factoriser `onlyOwner`, `whenNotPaused`, etc.


In [9]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract ModifierExample {
    address public owner;
    bool public paused = false;
    uint256 public counter = 0;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, ""Pas le proprietaire"");
        _;
    }

    modifier whenNotPaused() {
        require(!paused, ""Contrat en pause"");
        _;
    }

    function pause() public onlyOwner {
        paused = true;
    }

    function unpause() public onlyOwner {
        paused = false;
    }

    function increment() public whenNotPaused {
        counter += 1;
    }

    function criticalOperation() public onlyOwner whenNotPaused returns (uint256) {
        counter += 10;
        return counter;
    }
}
";

 var c = await Deploy(source,"ModifierExample");

Console.WriteLine($"paused = {await c.GetFunction("paused").CallAsync<bool>()}; counter = {await c.GetFunction("counter").CallAsync<BigInteger>()}");
 await Write(c,"increment");
 Console.WriteLine($"counter = {await c.GetFunction("counter").CallAsync<BigInteger>()}");
 await Write(c,"pause");

 // Rejouer le même appel en lecture pour vérifier la raison du revert du modifier.
 var callRequest = System.Text.Json.JsonSerializer.Serialize(new {jsonrpc="2.0", id=1, method="eth_call", @params=new object[]{new {from=deployer,to=c.Address,data=c.GetFunction("increment").GetData()},"latest"}});
 using var http = new System.Net.Http.HttpClient();
 using var response = await http.PostAsync(url, new System.Net.Http.StringContent(callRequest, System.Text.Encoding.UTF8, "application/json"));
 response.EnsureSuccessStatusCode();
 using var rpcResult = System.Text.Json.JsonDocument.Parse(await response.Content.ReadAsStringAsync());
 if (!rpcResult.RootElement.TryGetProperty("error",out var rpcError) || !rpcError.GetProperty("message").GetString().Contains("Contrat en pause")) throw new Exception("Raison du revert inattendue : " + rpcResult.RootElement.ToString());
 Console.WriteLine("Raison eth_call vérifiée : Contrat en pause");

 bool rejected=false;
 try { await Write(c,"increment");
 } catch(InvalidOperationException ex) when(ex.Message.StartsWith("Receipt status 0")) { rejected=true;
 Console.WriteLine($"Revert attendu : {ex.Message}");
 } if(!rejected) throw new Exception("La pause aurait dû interdire increment");
 await Write(c,"unpause");
 await Write(c,"criticalOperation");
 Console.WriteLine($"counter = {await c.GetFunction("counter").CallAsync<BigInteger>()}");

}

ModifierExample déployé : 0x02b0b4efd909240fcb2eb5fae060dc60d112e3a4, tx 0x8acefe92837904c5a7a33a77dbae355c851f7d9f60f1838d627dfd5854530e37, gas 422425


paused = False; counter = 0


increment : tx 0xc39dd7dab1c428f1aa0d66ad719c38b7b66cf275543c19640711bc02fed6976d, bloc 155, gas 45789


counter = 1


pause : tx 0x0deb7844b220542dfe0406400acd4880aabeaa66233bde2c5e3f0c3c80567c0f, bloc 156, gas 26513


Raison eth_call vérifiée : Contrat en pause


Revert attendu : Receipt status 0 : 0x3694f255e76cef6130687092157a48f076380f922141bf78358041ec24a1bd9e, bloc 157, gas 23762


unpause : tx 0x04570425f2454f90f8bf437ea368a87abe32a8ed1f1267fecdc80e344c9b6a5c, bloc 158, gas 26513


criticalOperation : tx 0x6bad37ed987874050095f63f7e71ee3795d7abe66dbdf6c66c2d4cfee9f94a7d, bloc 159, gas 29081


counter = 11


### Interpretation : Custom Modifiers - Reutilisation de logique

**Résultat obtenu** : Les modifiers encapsulent des conditions prealables reutilisables.

| Modifier | Condition | Effet |
|----------|-----------|-------|
| `onlyOwner` | `msg.sender == owner` | Restreint au propriétaire |
| `whenNotPaused` | `!paused` | Bloque si le contrat est en pause |
| `onlyOwner whenNotPaused` | Les deux conditions | Combine les restrictions |

**Points clés** :
- Le symbol `_;` est l'endroit ou le code de la fonction est exécuté (après les checks du modifier)
- `pause()` et `unpause()` ne peuvent être appeles que par le owner (grace a `onlyOwner`)
- `increment()` echoue quand le contrat est en pause (revert avec "Contrat en pause")
- `criticalOperation()` combine deux modifiers : vérification owner + état pause
- Les modifiers reduisent la duplication de code et augmentent la lisibilite

### Exercice 2 : Contrat Vault avec modifier onlyOwner

Mettez en pratique les **custom modifiers** (section 4) et **payable** (section 3.2).
Créez un coffre-fort ou seul le propriétaire peut retirer les fonds.

**Indice** : `require(msg.sender == owner, "Not owner"); _;` pour le modifier.

Compétence visée : combiner `payable` (recevoir des ETH) et un modifier `onlyOwner` (restreindre les retraits) dans un même contrat — c'est exactement le patron assemblé par les exemples guidés 1 et 2.


In [10]:
{
// Exercice 2 : Contrat Vault avec modifier onlyOwner
// TODO etudiant : implementer un coffre-fort owner-only
// Etape 1 : Definir address public owner, capture msg.sender dans le constructor
// Etape 2 : Definir modifier onlyOwner() avec require(msg.sender == owner); _;
// Etape 3 : Implementer deposit() public payable
// Etape 4 : Implementer withdrawAll() public onlyOwner
var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Vault {
    // TODO etudiant : owner, modifier onlyOwner, deposit, withdrawAll
}
";

// var c = await Deploy(source, "Vault");
// await WriteFrom(deployer, c, "deposit", eth);
// await Write(c, "withdrawAll");
Console.WriteLine("Exercice à compléter : Vault");

}

Exercice à compléter : Vault



(8,5): warning CS0219: The variable 'source' is assigned but its value is never used



***

## 5. Fonctions speciales

Trois fonctions sans nom usuel : le `constructor` (exécuté une seule fois, au déploiement), `receive()` (déclenché par un transfert ETH simple, sans data) et `fallback()` (appel d'une fonction inexistante, ou transfert avec data).


In [11]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SpecialFunctions {
    address public owner;
    string public name;
    uint256 public receivedTotal = 0;

    event Received(address from, uint256 amount);

    // CONSTRUCTOR : execute une seule fois au deploiement
    constructor(string memory _name) {
        owner = msg.sender;
        name = _name;
    }

    // RECEIVE : appele lors d'un transfert ETH simple (sans data)
    receive() external payable {
        receivedTotal += msg.value;
        emit Received(msg.sender, msg.value);
    }

    function getBalance() public view returns (uint256) {
        return address(this).balance;
    }
}
";

 // Une transaction sans données appelle receive. Contrôler son receipt avant de relire le solde.
var c = await Deploy(source,"SpecialFunctions","MonContrat");

Console.WriteLine($"name = {await c.GetFunction("name").CallAsync<string>()}; owner = {await c.GetFunction("owner").CallAsync<string>()}; total = {await c.GetFunction("receivedTotal").CallAsync<BigInteger>()}");
 var receipt=await web3.Eth.TransactionManager.SendTransactionAndWaitForReceiptAsync(new Nethereum.RPC.Eth.DTOs.TransactionInput {From=deployer,To=c.Address,Value=new HexBigInteger(eth/10),Gas=new HexBigInteger(100000)});
 if(receipt.Status.Value!=1 || receipt.BlockNumber==null) throw new Exception("Transfert non miné");
 Console.WriteLine($"receive tx {receipt.TransactionHash}, gas {receipt.GasUsed.Value}; total = {await c.GetFunction("receivedTotal").CallAsync<BigInteger>()}; balance = {await c.GetFunction("getBalance").CallAsync<BigInteger>()} Wei");

}

SpecialFunctions déployé : 0x0ed64d01d0b4b655e410ef1441dd677b695639e7, tx 0x44ec34d7b35faa6a4bc0545cd7cddaa0d0c51655ab7b62228169c2e05679cb49, gas 353559


name = MonContrat; owner = 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266; total = 0


receive tx 0xfe88e1346bd55d466664e3e0ec5c7b0604c9401a34e9c40303153b8a326f5505, gas 44957; total = 100000000000000000; balance = 100000000000000000 Wei


### Interpretation : Fonctions speciales - Constructor et Receive

**Résultat obtenu** : Le constructor initialise le contrat UNE SEULE FOIS, receive est appele automatiquement lors d'un transfert ETH.

| Fonction | Quand est-elle appelee ? | Usage |
|----------|------------------------|-------|
| `constructor` | Une seule fois au déploiement | Initialisation (owner, name) |
| `receive()` | Transfert ETH simple (sans data) | Reception automatique d'ETH |
| `fallback()` | Appel de fonction inexistante | Gestion d'erreur |

**Points clés** :
- Le constructor peut prendre des arguments (`_name` dans l'exemple)
- `receive()` est `external payable` : ne peut pas être appele directement, seulement via transfert ETH
- L'event `Received` est logge lors de la reception d'ETH
- `receivedTotal` compte le total des ETH recus via `receive()`

### Exercice 3 : Contrat Registry (data locations storage vs memory)

Mettez en pratique les **data locations** (section 1). Le piege : `User memory u`
créé une copie (modifications perdues), `User storage u` modifie réellement le tableau.

**Indice** : `User storage u = users[index];` pour une vraie reference storage.

Le piège visé est celui de la section 1 : `User memory u = users[index]` crée une copie dont les modifications sont perdues à la fin de l'appel, tandis que `User storage u = users[index]` référence directement le storage et modifie réellement le tableau.


In [12]:
{
// Exercice 3 : Contrat Registry (data locations storage vs memory)
// TODO etudiant : implementer un registre ou bumpScore modifie le storage (pas une copie memory)
// Etape 1 : Definir struct User { string name; uint256 score; } et User[] public users
// Etape 2 : Implementer addUser(string memory name)
// Etape 3 : Implementer bumpScore(uint256 index) avec User storage u = users[index];
// Etape 4 : Implementer getScore(uint256 index) public view returns (uint256)
var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Registry {
    // TODO etudiant : struct User, tableau users, addUser, bumpScore (storage ref), getScore
}
";

// var c = await Deploy(source, "Registry");
// await Write(c, "addUser", "Alice");
// await Write(c, "bumpScore", new BigInteger(0));
// await Write(c, "bumpScore", new BigInteger(0));
// Console.WriteLine(await c.GetFunction("getScore").CallAsync<BigInteger>(new BigInteger(0)));
Console.WriteLine("Exercice à compléter : Registry");

}

Exercice à compléter : Registry



(8,5): warning CS0219: The variable 'source' is assigned but its value is never used



***

## 6. Exemples guidés

Chaque exemple guidé sert de patron : l'énoncé pose le contrat à construire, et la cellule suivante déploie une version de référence sur anvil dont la sortie commise montre le comportement attendu. Les exercices 3 à 5, plus loin dans ce notebook, vous demandent de reconstruire ce comportement vous-même.


### Exemple guide 1 : Contrat Bank

Créez un contrat qui permet de deposer et retirer des ETH avec un modifier `onlyOwner`.

Compétence visée : assembler `payable` (recevoir) et un modifier `onlyOwner` (restreindre le retrait) dans un même contrat, comme le fera le coffre-fort de l'exercice 3.


In [13]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Bank {
    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, ""Not owner"");
        _;
    }

    function deposit() public payable {
        balances[msg.sender] += msg.value;
    }

    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, ""Insufficient funds"");
        balances[msg.sender] -= amount;
        (bool success, ) = payable(msg.sender).call{value: amount}("""");
        require(success, ""Transfer failed"");
    }

    function emergencyWithdraw() public onlyOwner {
        (bool success, ) = payable(owner).call{value: address(this).balance}("""");
        require(success, ""Transfer failed"");
    }
}
";

 // Le deuxième compte dépose et retire ; le déployeur seul effectue le retrait d’urgence.
var c = await Deploy(source,"Bank");

var client=accounts[1];
 await WriteFrom(client,c,"deposit",2*eth);
 Console.WriteLine($"dépôt = {await c.GetFunction("balances").CallAsync<BigInteger>(client)} Wei");
 await WriteFrom(client,c,"withdraw",BigInteger.Zero,3*eth/4);
 Console.WriteLine($"retrait = {await c.GetFunction("balances").CallAsync<BigInteger>(client)} Wei; contrat = {(await web3.Eth.GetBalance.SendRequestAsync(c.Address)).Value}");
 await Write(c,"emergencyWithdraw");
 Console.WriteLine($"après emergencyWithdraw = {(await web3.Eth.GetBalance.SendRequestAsync(c.Address)).Value} Wei");

}

Bank déployé : 0x40a42baf86fc821f972ad2ac878729063ceef403, tx 0x0fecf8e899f0236b4a188a245e416359ab7a2c90e733b6c60ec0a45f3012e385, gas 512123


deposit : tx 0x76997d8a621a33a3c1888977510d1f2f780fc8302a474c3fe220fbf711976a2f, bloc 163, gas 43624


dépôt = 2000000000000000000 Wei


withdraw : tx 0x92c35a9a0f4dcbc2d942b30f40203551abc6201dfb89241a60bb291204441e44, bloc 164, gas 34279


retrait = 1250000000000000000 Wei; contrat = 1250000000000000000


emergencyWithdraw : tx 0x459c48cd81f26a2fafb3823f16dc5850a8591375e03ff0de528cdc7ef5f11ebb, bloc 165, gas 30681


après emergencyWithdraw = 0 Wei


**Indice** : Pour `onlyOwner`, utilisez `require(msg.sender == owner, "Not owner"); _;`. Pour `deposit`, faites `balances[msg.sender] += msg.value`. Pour `withdraw`, vérifiez les fonds avec `require`, debitez le solde, et utilisez `payable(msg.sender).call{value: amount}("")`.

### Exemple guide 2 : Contrat Pausable

Créez un contrat avec des fonctions qui peuvent être mises en pause.

Compétence visée : protéger une fonction par un état (`paused`) avec le modifier `whenNotPaused`, et rendre l'administration (`pause`/`unpause`) réservée au propriétaire.


In [14]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Pausable {
    address public owner;
    bool public paused = false;
    uint256 public counter = 0;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, ""Not owner"");
        _;
    }

    modifier whenNotPaused() {
        require(!paused, ""Paused"");
        _;
    }

    function pause() public onlyOwner {
        paused = true;
    }

    function unpause() public onlyOwner {
        paused = false;
    }

    function increment() public whenNotPaused {
        counter += 1;
    }

    function reset() public onlyOwner whenNotPaused {
        counter = 0;
    }
}
";

 var c = await Deploy(source,"Pausable");

Console.WriteLine($"paused = {await c.GetFunction("paused").CallAsync<bool>()}; counter = {await c.GetFunction("counter").CallAsync<BigInteger>()}");
 await Write(c,"increment");
 Console.WriteLine($"counter = {await c.GetFunction("counter").CallAsync<BigInteger>()}");
 await Write(c,"pause");

 // Rejouer le même appel en lecture pour vérifier la raison du revert du modifier.
 var callRequest = System.Text.Json.JsonSerializer.Serialize(new {jsonrpc="2.0", id=1, method="eth_call", @params=new object[]{new {from=deployer,to=c.Address,data=c.GetFunction("increment").GetData()},"latest"}});
 using var http = new System.Net.Http.HttpClient();
 using var response = await http.PostAsync(url, new System.Net.Http.StringContent(callRequest, System.Text.Encoding.UTF8, "application/json"));
 response.EnsureSuccessStatusCode();
 using var rpcResult = System.Text.Json.JsonDocument.Parse(await response.Content.ReadAsStringAsync());
 if (!rpcResult.RootElement.TryGetProperty("error",out var rpcError) || !rpcError.GetProperty("message").GetString().Contains("Paused")) throw new Exception("Raison du revert inattendue : " + rpcResult.RootElement.ToString());
 Console.WriteLine("Raison eth_call vérifiée : Paused");

 bool rejected=false;
 try { await Write(c,"increment");
 } catch(InvalidOperationException ex) when(ex.Message.StartsWith("Receipt status 0")) { rejected=true;
 Console.WriteLine($"Revert attendu : {ex.Message}");
 } if(!rejected) throw new Exception("La pause aurait dû interdire increment");
 await Write(c,"unpause");
 await Write(c,"reset");
 Console.WriteLine($"paused = {await c.GetFunction("paused").CallAsync<bool>()}; counter = {await c.GetFunction("counter").CallAsync<BigInteger>()}");

}

Pausable déployé : 0x986aaa537b8cc170761fdac6ac4fc7f9d8a20a8c, tx 0x87d067b5b6aeb060f55ff23428221ddfd66457143723959e767533ecfdacacc5, gas 412433


paused = False; counter = 0


increment : tx 0xe0d69e79426ff7afd8bec36b3a52ab6e37627c490758eef1a41565d11d49750a, bloc 167, gas 45767


counter = 1


pause : tx 0x11ec8fe7a64cb9a69f3b087e50c36e00ff245daca2c12d2d9a70cf62a8627586, bloc 168, gas 26491


Raison eth_call vérifiée : Paused


Revert attendu : Receipt status 0 : 0xcb708b8e016c843367fc9da963eaf2ccdb8fe812ba25ebf2866f3ba0baa07782, bloc 169, gas 23740


unpause : tx 0x9197f690244c3177d46ff71822b425ccd5b2f9c15547fd98226d83cf7a3b7438, bloc 170, gas 26491


reset : tx 0xbfa64798ef774b3f5257b652837b7176e311383fc783b339439cbb9cc3c2863a, bloc 171, gas 23859


paused = False; counter = 0


**Indice** : Initialisez `owner` dans le constructor. Pour les modifiers, utilisez `require` suivi de `_;`. Les fonctions `pause` et `unpause` doivent simplement modifier `paused`. `increment` et `reset` sont proteges par `whenNotPaused`.

## Comprendre les transactions et leurs effets

Les fonctions `view` et `pure` sont interrogées avec `eth_call` : le nœud exécute le code sur une copie de l’état, sans créer de transaction. Une fonction qui écrit ou reçoit des fonds exige une transaction. Le client transmet l’adresse de l’émetteur, la limite de gas, les arguments ABI et, pour un dépôt payable, la valeur en Wei. Le hash seul ne prouve pas que l’écriture a réussi : le receipt donne le bloc, le gas utilisé et un statut de succès ou d’échec. Le code attend ce receipt avant toute lecture postérieure.

Le contrat Bank distingue le compte du propriétaire et celui du client. Le dépôt est crédité au client qui envoie la transaction ; son retrait réduit à la fois son solde enregistré et le solde du contrat. Le retrait d’urgence part du propriétaire. Ces identités ne sont pas interchangeables : elles donnent sa signification au test des modifiers. Les comptes de cette démonstration sont déverrouillés dans Anvil, jamais des comptes de production.

Les deux estimations de gas comparent les mêmes cinq entiers, avec le même compilateur et le même nœud. Elles répondent à une question circonscrite : le coût estimé de ces deux fonctions sur ces données. Elles ne constituent pas une loi universelle sur toutes les fonctions `public` et `external`, car l’optimiseur, les paramètres et le travail effectué influencent le coût. Une lecture `eth_call` n’engage pas de frais pour le demandeur ; le gas estimé permet ici de comparer le travail que coûterait une transaction.

## Vérifiez votre compréhension

1. Pourquoi `updateBalanceBad` ne change-t-il pas la valeur enregistrée pour Alice ?
2. Que signifie un receipt dont le statut vaut zéro pendant la pause ?
3. Pourquoi faut-il envoyer le dépôt Bank depuis le compte du client ?
4. Que compare la mesure public/external du carnet ?

<details><summary>Éléments de réponse</summary>

1. Elle modifie une copie `memory`, tandis que `updateBalanceGood` utilise une référence `storage`.
2. La transaction a été minée mais son exécution a échoué ; la modification demandée n’a pas été conservée. La pause est activée avant cet essai et levée ensuite.
3. Le contrat crédite `balances[msg.sender]` ; l’émetteur détermine donc quel compte reçoit le crédit et peut le retirer.
4. Elle estime les deux fonctions sur le même tableau et le même nœud. La différence affichée dépend de cette compilation et de ces entrées.

</details>

## Pour aller plus loin

En poursuivant vers l’héritage, examinez comment un modifier et ses contrôles se composent avec les fonctions d’un contrat enfant. Dans les exercices, complétez les squelettes Solidity avant de décommenter les appels C# et comparez les résultats lus après chaque receipt.

***

## 7. Résumé

| Concept | Description |
|---------|-------------|
| `storage` | Données permanentes sur blockchain |
| `memory` | Données temporaires en RAM |
| `calldata` | Données d'entrée read-only |
| `public` | Accessible partout |
| `external` | Accessible uniquement de l'exterieur |
| `internal` | Ce contrat et enfants |
| `private` | Uniquement ce contrat |
| `view` | Lecture seule du storage |
| `pure` | Aucun accès au storage |
| `payable` | Peut recevoir des ETH |

***

**Notebook suivant** : [SC-05-Inheritance-Python](SC-05-Inheritance-Python.ipynb)

Au-delà du tableau : retenez que la data location décide du coût et de la persistance (`storage` permanent, `memory` éphémère, `calldata` lecture seule), que la visibilité organise les appels sans rien cacher (tout est lisible sur la chaîne), et que `view`/`pure` rendent les lectures gratuites tandis que toute écriture reste une transaction payante.


## À retenir

Les emplacements de données et la visibilité déterminent le comportement des fonctions. Les écritures et transferts exigent des transactions minées ; `view` et `pure` peuvent être interrogées sans écriture.

[Version Python](SC-04-Functions-State-Python.ipynb) | [Solidity Basics](SC-03-Solidity-Basics-CSharp.ipynb)